# Chapter 10: Who paid whom

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "10"
__file__ = os.path.abspath("run.py")

## The scene

Thursday 23 July, 10:10. Varma has not forgotten April. Three days of tracing, a compacted
history, and a summary that kept the amounts and lost which account paid which.

"It summarised the trail into a list of numbers," she says. "Tell me that cannot happen again."

You cannot tell her that about a paragraph the model writes freely. You can about a form.

Okafor drafts it on the whiteboard as he would a schedule: payer, payee, total, count, the
statement it came from. Open questions. Accounts read.

"If a line has no payer," he says, "it is not a line."

## The concept

**Compaction** replaces the turns of a long conversation with something shorter, so the agent
can carry on inside its window. Everything after the task is thrown away and one message takes
its place. Whatever that message leaves out, the agent no longer knows.

Case file 02 compacted to a paragraph. A paragraph has no place where a payer must go, and the
model that writes it is the model that will have nothing else to read afterwards. A **schema**
gives the compacted state named fields:

- **flows**, each with a payer, a payee, a number of payments, a total and the statement it
  came from;
- **accounts read**, every file opened;
- **open questions**.

A schema on its own only says what should be there. The other half is a **check in code** after
every compaction. Code keeps a **ledger** of every tool result outside the window, compares the
form with it, and refuses a form that drops a flow, changes a figure, invents a flow or leaves a
file out. A refused form goes back to the model with the list of what was wrong.

The form can also be filled in by code, straight from the ledger. Then no model writes it at
all. That keeps every fact. It does not make the agent use them, and this chapter measures the
difference.

```mermaid
flowchart TD
    R[Tool results] --> L[(Ledger, outside the window)]
    C{{Conversation past 1,500 tokens}} --> P[A paragraph the model writes]
    C --> F[A form the model fills in]
    C --> B[A form code fills in from the ledger]
    F --> K{check against the ledger}
    L --> K
    L --> B
    K -- problems --> F
    K -- none --> S[Compacted state replaces the turns]
    P --> S
    B --> S
    S --> A[Agent carries on and reports]
    A --> V{report_covers: every flow in the ledger?}
    L --> V
    V -- flows left out --> A
    V -- all there --> D[Report accepted]
```

## How it works

1. **The form.** `lantern.compaction.SCHEMA`: flows with five required fields, the files read,
   the open questions. The model is asked for it with structured output.
2. **A paragraph, as in April.** The checked trace of chapter 09, with the digest tool, is
   compacted whenever the conversation passes 1,500 tokens, by asking the model for one
   paragraph. After each compaction, `compaction.survived()` counts the flows from the ledger
   that are still whole in it: payer, payee and total, the total as a whole figure.
3. **A form, checked in code.** The model fills in the form; `compaction.check()` compares it
   with the ledger. A refused form is asked for again with its problems; after three refused
   attempts the form is filled in from the ledger.
4. **The form filled in by code.** `compaction.from_source()`: the form built from the ledger
   at every compaction, no model involved.
5. **The report, checked against the ledger.** The same, and the final report is also checked
   by `compaction.report_covers()`, which sends it back while it leaves out a flow the ledger
   holds (Northwind Freight, which the task says to ignore, excepted).

The check, in outline:

```python
def check(state, ledger):
    want = {(f["payer"], f["payee"]): f for f in seen_flows(ledger)}   # no bank charges
    returned = all_pairs(ledger)                                       # banks included
    ...
    for key, f in want.items():                  # dropped, wrong total, wrong count
        ...
    for key in got:                              # invented, whoever the payee
        if key not in returned:
            problems.append(f"not in any tool result: {key[0]} paid {key[1]}")
    unread = sorted(set(ledger) - set(state.get("accounts_read", [])))
    ...
```

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import json

from lantern import agent, compaction, context, learn, trail
from lantern.config import ROOT, load_config
from lantern.llm import chat
from lantern.ui import Table, chapter, console, evidence, finding, note, reply, save_shot, step

chapter(
    "10",
    "Who paid whom",
    """
Thursday 23 July, 10:10. Varma has not forgotten April. Three days of tracing, a compacted
history, and a summary that kept the amounts and lost which account paid which.

"It summarised the trail into a list of numbers," she says. "Tell me that cannot happen again."

You cannot tell her that about a paragraph the model writes freely. You can about a form.

Okafor drafts it on the whiteboard as he would a schedule: payer, payee, total, count, the
statement it came from. Open questions. Accounts read.

"If a line has no payer," he says, "it is not a line."
""",
)

MINE = learn.mine()  # --mine: your check() from practice/exercise.py
if MINE:
    compaction.check = MINE.check
OUT = ROOT / ".scratch" / "10"
RESERVE = 600
MAX_TURNS = 18
# The conversation is compacted when it passes this many tokens. Challenge: lower it and count
# the compactions; summaries of summaries are where a free-text compaction loses most.
THRESHOLD = 1500


def trace(make_hook=None, report_checked: bool = False):
    """One checked trace. With `report_checked`, the report must also hold every flow in the
    ledger of tool results, or it is sent back."""
    text, names = trail.listing(True)
    schemas, handlers, opened = trail.tools(names)
    log = compaction.Log()
    ledger: dict = {}
    hook = make_hook(log, ledger) if make_hook else None
    followed = trail.not_done(opened)
    covers = compaction.report_covers(ledger, skip=("Northwind Freight Ltd",))

    def until(report: str):
        return followed(report) or (covers(report) if report_checked else None)

    run = agent.run([{"role": "system", "content": trail.RULES + text},
                     {"role": "user", "content": trail.TASK}], chat=chat, schemas=schemas,
                    handlers=handlers, max_turns=MAX_TURNS, max_tokens=RESERVE,
                    limit=window - RESERVE, until=until, before_request=hook)
    return {"run": run, "opened": list(dict.fromkeys(opened)), "grade": trail.grade(run.answer),
            "log": log}


OUT.mkdir(parents=True, exist_ok=True)
window = load_config()["generation"]["num_ctx"]
want = trail.expected()
count = context.count_tokens

### 1. The form

In [ ]:
step("1. The form")
evidence("lantern.compaction.SCHEMA, as the model is asked to fill it in",
         '{"flows": [{"payer", "payee", "payments", "total", "source"}], '
         '"accounts_read": [file], "open_questions": [text]}')
note("Every flow needs all five fields. The check compares each one with the tool result it "
     "came from, kept in a ledger outside the window, and names what is missing or wrong.")

### 2. A paragraph, as in April

In [ ]:
step("2. A paragraph, as in April")
p_para = learn.predict(
    f"The trace is compacted whenever the conversation passes {THRESHOLD:,} tokens, by asking "
    "the model for one paragraph. Of the flows the tools had returned by the last compaction, "
    "how many are still whole in the paragraph: payer, payee and total?",
    ["All of them", "More than half", "Half or fewer", "None"])
arms = {}
arms["no compaction"] = trace()
arms["a paragraph"] = trace(
    lambda log, ledger: compaction.free_text(chat, THRESHOLD, count, log, ledger))
para = arms["a paragraph"]
table = Table(title="A paragraph: what each compaction kept")
for col in ("compaction", "flows the tools had returned", "still whole in the paragraph"):
    table.add_column(col, justify="right")
for i, (kept, seen) in enumerate(para["log"].kept, 1):
    table.add_row(str(i), str(seen), str(kept))
console.print(table)
if para["log"].states:
    reply("the last paragraph LANTERN wrote for itself", para["log"].states[-1][:600])
last_kept, last_seen = para["log"].kept[-1] if para["log"].kept else (0, 0)
note(f"{para['log'].compactions} compactions. After the last, {last_kept} of {last_seen} flows "
     f"were whole. The trace then reported {para['grade']['found']} of {len(want)} hops, totals "
     f"right on {para['grade']['totals_right']}.")
learn.reveal(p_para, answer=0 if last_kept == last_seen else 1 if last_kept > last_seen / 2
             else 2 if last_kept else 3,
             why="A paragraph has no place a payer must go. The model that writes it decides "
             "what matters, and it is the same model that will have nothing else to read.")

### 3. A form, checked in code

In [ ]:
step("3. A form, checked in code")
p_form = learn.predict(
    "Now the model fills in the form, and code refuses a form with a missing field or a figure "
    "that differs from the tool's. How many times is a form refused over the run?",
    ["Never", "Once or twice", "3 to 5 times", "More than 5"])
arms["a form, checked"] = trace(
    lambda log, ledger: compaction.with_schema(chat, THRESHOLD, count, log, ledger=ledger))
form = arms["a form, checked"]
for problems in form["log"].problems[:3]:
    reply("a form refused", "; ".join(problems)[:400], style="red")
note(f"{form['log'].compactions} compactions, {form['log'].refused} forms refused. Every form "
     f"that was kept had every flow whole: {form['log'].kept}.")
learn.reveal(p_form, answer=learn.bucket(form["log"].refused, [0, 2, 5]),
             why="The check does not make the model careful. It makes carelessness visible, "
             "and gives the next attempt the list of what was wrong.")

### 4. The form filled in by code

In [ ]:
step("4. The form filled in by code")
arms["a form, filled in by code"] = trace(
    lambda log, ledger: compaction.from_source(THRESHOLD, count, log, ledger))
code = arms["a form, filled in by code"]
reply("the report after a form that kept every flow", code["run"].answer[:400])
note(f"The form was whole: {code['log'].kept}. The report holds {code['grade']['found']} of "
     f"{len(want)} hops. LANTERN reported what it had read since the compaction and treated "
     "the form as background. Keeping a fact and using it are separate things.")

### 5. The report, checked against the ledger

In [ ]:
step("5. The report, checked against the ledger")
p_cov = learn.predict(
    "Last arm. The form is filled in by code, and the final report is also checked in code: "
    "it is sent back while it leaves out any flow in the ledger. How many hops now?",
    ["All 7", "5 or 6", "1 to 4", "None"])
arms["a form by code, report checked"] = trace(
    lambda log, ledger: compaction.from_source(THRESHOLD, count, log, ledger),
    report_checked=True)
both = arms["a form by code, report checked"]
learn.reveal(p_cov, answer=0 if both["grade"]["found"] == 7 else 1 if both["grade"]["found"] >= 5
             else 2 if both["grade"]["found"] else 3,
             why="The compaction keeps the facts. A second check makes the report carry them. "
             "Neither check is the model being careful.")
table = Table(title="The same checked trace, five ways")
for col in ("compaction", "hops found", "totals right", "not hops", "compactions", "refused",
            "requests", "tokens, all requests", "largest request", "stopped"):
    table.add_column(col)
for name, a in arms.items():
    g, run, log = a["grade"], a["run"], a["log"]
    table.add_row(name, f"{g['found']} of {len(want)}", str(g["totals_right"]),
                  str(g["invented"]), str(log.compactions), str(log.refused),
                  str(len(run.turns)), f"{run.total_prompt_tokens:,}",
                  f"{run.largest_request:,}", run.stopped.split(":")[0])
console.print(table)
(OUT / "traces.json").write_text(json.dumps(
    {name: {"answer": a["run"].answer, "grade": a["grade"], "stopped": a["run"].stopped,
            "states": a["log"].states, "problems": a["log"].problems, "kept": a["log"].kept}
     for name, a in arms.items()}, indent=2), encoding="utf-8")
note("The last two rows have no model in the compaction at all: the form is built from the "
     "ledger of tool results. It cannot drop a payer, and it cannot record an open question, "
     "because nothing in a tool result is a question.")


def m(arm, key):
    return arms[arm]["grade"][key]


learn.record(
    hops=len(want),
    threshold=THRESHOLD,
    plain_found=m("no compaction", "found"),
    plain_tokens=arms["no compaction"]["run"].total_prompt_tokens,
    plain_largest=arms["no compaction"]["run"].largest_request,
    paragraph_found=m("a paragraph", "found"),
    paragraph_totals=m("a paragraph", "totals_right"),
    paragraph_compactions=para["log"].compactions,
    paragraph_kept_last=last_kept,
    paragraph_seen_last=last_seen,
    paragraph_stopped=para["run"].stopped.split(":")[0],
    form_found=m("a form, checked", "found"),
    form_totals=m("a form, checked", "totals_right"),
    form_compactions=form["log"].compactions,
    form_refused=form["log"].refused,
    form_largest=form["run"].largest_request,
    form_stopped=form["run"].stopped.split(":")[0],
    code_found=m("a form, filled in by code", "found"),
    code_totals=m("a form, filled in by code", "totals_right"),
    code_compactions=code["log"].compactions,
    code_tokens=code["run"].total_prompt_tokens,
    code_largest=code["run"].largest_request,
    code_kept_last=code["log"].kept[-1][0] if code["log"].kept else 0,
    both_found=both["grade"]["found"],
    both_totals=both["grade"]["totals_right"],
    both_invented=both["grade"]["invented"],
    both_sent_back=both["run"].sent_back,
    both_tokens=both["run"].total_prompt_tokens,
    both_largest=both["run"].largest_request,
    both_stopped=both["run"].stopped.split(":")[0],
    mine=MINE is not None,
)

finding(f"""
Compacted to a paragraph whenever it passed {THRESHOLD:,} tokens, the trace was compacted
{para['log'].compactions} times; after the last, {last_kept} of the {last_seen} flows the tools
had returned were still whole, and the report held {m('a paragraph', 'found')} of {len(want)}
hops with {m('a paragraph', 'totals_right')} totals right. With a form checked in code,
{form['log'].refused} forms were refused and the report held {m('a form, checked', 'found')}
hops with {m('a form, checked', 'totals_right')} totals right. With the form filled in by code
from the tool results, the form held every flow and the report
{m('a form, filled in by code', 'found')} hops. With the report also checked against the ledger
and sent back {both['run'].sent_back} times:
{both['grade']['found']} of {len(want)} hops, totals right on {both['grade']['totals_right']}, in
{both['run'].total_prompt_tokens:,} tokens with no request over {both['run'].largest_request:,},
where the uncompacted trace took {arms['no compaction']['run'].total_prompt_tokens:,} and reached
{arms['no compaction']['run'].largest_request:,}.
""")
save_shot("10-compaction-with-a-schema")

## The nuances

- **The summary of a summary lost the trail.** The first paragraph kept 5 of the 6 flows the
  tools had returned; the one it dropped was the payment to Northwind, which the task says to
  ignore. The second was written from the first and the turns since, and kept 1 of 8: one line,
  Osprey Point to Tidewater, and DONE. The report then held 1 of 7 hops and named Sabran
  Commercial Bank as a payer of Osprey Point.
- **The paragraph was not a paragraph.** Asked for one paragraph, LANTERN wrote FLOW lines both
  times. The format of a free summary is the model's choice, and so is what it keeps.
- **The check never had to refuse.** Asked for the form with structured output, LANTERN filled
  it in correctly both times: 6 of 6 flows whole, then 7 of 7, and every file listed. 0 forms
  were refused.
- **A whole form and an empty report.** The same arm reported 0 of 7 hops. Its report was four
  bank charges, all four copied from its own last form, which listed them after the hops. The
  task says to ignore bank charges. The check proves the facts survived the compaction; it does
  not prove the agent reports from them.
- **Filled in by code, the same.** With the form built from the ledger, every flow was in it
  and the report held 0 of 7 hops: one line, Tidewater's bank charge, read after the
  compaction. LANTERN treated the form as background.
- **A second check made the report carry the facts.** With the report also compared with the
  ledger, it was sent back 8 times and reached 6 of 7 hops, all six totals right, before the
  turn limit of 18 requests. The hop it left out, Greyfen to Pellmar, was in the last form.
- **The check proves what is present, not that nothing else was lost.** The form has no place
  for why an account was opened. The model's first form held one open question, "Is there any
  further flow from Osprey Point Management Ltd?"; its second held none, and nothing checked
  that. A form filled in by code never has one, because nothing in a tool result is a question.
- **Compaction kept the requests small.** The uncompacted trace reached 2,851 tokens in one
  request; the compacted arms stayed between 1,594 and 1,788. The checked arm spent about as
  many tokens in all (23,228 against 23,899), because being sent back costs requests.

## Try this

1. Open `.scratch/10/traces.json` and compare the two paragraphs in "a paragraph". What did
   the second have to work from that the first did not?
2. In "a form, checked", put the last form beside the report. Which lines of the report come
   from the form, and where in the form are they?
3. Remove `source` from the form and from `check`. Which problems can the check no longer name?

## Master it

After this chapter you can compact a conversation into a form with named fields, check a
compacted state in code against a ledger of tool results kept outside the window, refuse a
form and say why, and explain why a state that keeps every fact still needs a check on what the
agent reports from it.

**Predict, then see.** `make lesson N=10` stops three times: how many flows survive the last
paragraph, how often a form is refused, and how many hops the report holds once it is checked.

**Build it.** Write `number` and `check` in [`practice/exercise.py`](practice/exercise.py):

```bash
make exercise N=10          # 9 offline tests
make hint N=10
make lesson N=10 ARGS=--mine
```

**Check yourself.** `make quiz N=10`: seven questions.

**Break it.** `make challenge N=10`: your check on every form; compacting more often; and the
last arm without its report check.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 10

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 10

In [ ]:
%run ../../scripts/learn.py hint 10

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 10

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

Compacted to a paragraph whenever it passed 1,500 tokens, the trace was compacted twice; the
second paragraph, written from the first, kept 1 of the 8 flows the tools had returned, and the
report held 1 of 7 hops. Asked for a form and checked in code, LANTERN filled it in correctly
both times and no form was refused; its report held 0 of 7 hops and four bank charges. With the
form filled in by code it was the same: every flow kept, no hop reported. Only when the report
was also checked against the ledger, and sent back 8 times, did it reach 6 of 7 hops with every
total right, in 23,228 tokens with no request over 1,788. A compaction can now be made to keep
who paid whom. What LANTERN does with what was kept is a separate question, and the form has no
room for anything that is not a figure.

**Next:** [Chapter 11: Its own notes](../11-notes-and-scratchpads/README.md)